In [80]:
from sentence_transformers import (
  CrossEncoder,
  InputExample,
  losses,
  evaluation,
  SentenceTransformer,
  models,
  trainer,
  SentenceTransformerTrainer,
  SentenceTransformerTrainingArguments
  )
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator, CEBinaryAccuracyEvaluator
from datasets import Dataset
import math
import json
import torch
from torch.utils.data import DataLoader
from datetime import datetime
import random
import math

In [85]:
# training configs
data_ratio = 1
train_batch_size = 256
eval_batch_size = 64
num_epochs = 10
warmup_steps = math.ceil(len(train) * num_epochs * 0.1)

In [82]:
# organize the data
with open('data/train.json', 'r') as f:
    train = json.load(f)
random.shuffle(train)

with open('data/cv.json', 'r') as f:
    cv = json.load(f)
random.shuffle(cv)

with open('data/test.json', 'r') as f:
    test = json.load(f)
random.shuffle(cv)

train_data = [InputExample(texts=x, label=y) for [x, y] in train[0:math.floor(len(train) * data_ratio)]]
cv_data = [InputExample(texts=x, label=y) for [x, y] in cv[0:math.floor(len(cv) * data_ratio)]]
test_data = [InputExample(texts=x, label=y) for [x, y] in test[0:math.floor(len(test) * data_ratio)]]

print(train_data[0].texts)

['finger puppet', 'move']


## CrossEncoder Model - ms-marco-MiniLM-L-6-v2

In [89]:
# initialize the model
CE_model = CrossEncoder(
  "cross-encoder/ms-marco-MiniLM-L-6-v2",
  device="mps" if torch.backends.mps.is_available() else "cpu",
  default_activation_function=torch.nn.Sigmoid()
)

In [90]:
CE_model_save_path = "output/training_CE_" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# run the model
CE_model.fit(
    train_dataloader=DataLoader(
        dataset=train_data,
        shuffle=True,
        batch_size=train_batch_size,
        pin_memory=True),
    evaluator=CEBinaryClassificationEvaluator.from_input_examples(cv_data),
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    output_path=CE_model_save_path
)

# write the configs
with open(CE_model_save_path + "/config.txt", 'w') as f:
    f.write("train batch size: " + str(train_batch_size))
    f.write("num epochs: " + str(num_epochs) + "\n")
    f.write("warmup_steps: " + str(warmup_steps) + "\n")
    f.write("data_ratio: " + str(1 / data_ratio) + "\n")

Epoch:   0%|          | 0/10 [00:00<?, ?it/s]

Iteration:   0%|          | 0/837 [00:00<?, ?it/s]

Iteration:   0%|          | 0/837 [00:00<?, ?it/s]

Iteration:   0%|          | 0/837 [00:00<?, ?it/s]

Iteration:   0%|          | 0/837 [00:00<?, ?it/s]

...

Iteration:   0%|          | 0/837 [00:00<?, ?it/s]

Iteration:   0%|          | 0/837 [00:00<?, ?it/s]

Iteration:   0%|          | 0/837 [00:00<?, ?it/s]

Iteration:   0%|          | 0/837 [00:00<?, ?it/s]

Iteration:   0%|          | 0/837 [00:00<?, ?it/s]

In [134]:
# ... or load already trained model from local files
CE_model = CrossEncoder('output/final', default_activation_function=torch.nn.Sigmoid())

In [135]:
# positive cases
positive = [
  ["place of origin", "birthplace"],
  ["alien", "foreigner"],
  ["warship", "naval war vessel"],
  ["lemon treats", "lemon-flavored candy"],
  ["TikTok influencer", "someone who is famous on tiktok"],
  ["conflict", "drama"],
  ["clothing worn on the feet","socks"],
  ["throughout","always"],
  ["a break", "vacation"]
]
negative = [
  ["ice cream", "fondue"],
  ["warship", "Statue of Liberty"],
  ["birdfeed", "the study of birds"],
  ["staff made of magic", "jazz-fusion"]
]

for x in positive:
  similarity = CE_model.predict(x)
  print(x, ", ", similarity)
for x in negative:
  similarity = CE_model.predict(x)
  print(x, ", ", similarity)

['place of origin', 'birthplace'] ,  0.9910773
['alien', 'foreigner'] ,  0.9964958
['warship', 'naval war vessel'] ,  0.96390057
['lemon treats', 'lemon-flavored candy'] ,  0.9721844
['TikTok influencer', 'someone who is famous on tiktok'] ,  0.939585
['conflict', 'drama'] ,  0.99073505
['clothing worn on the feet', 'socks'] ,  0.9788255
['throughout', 'always'] ,  0.9946185
['a break', 'vacation'] ,  0.97737145
['ice cream', 'fondue'] ,  0.812898
['warship', 'Statue of Liberty'] ,  0.17632905
['birdfeed', 'the study of birds'] ,  0.47390643
['staff made of magic', 'jazz-fusion'] ,  0.28466183


pre-training results:
```
['place of origin', 'birthplace'] ,  0.00043210216
['alien', 'foreigner'] ,  0.0002477122
['warship', 'naval war vessel'] ,  0.01252699
['lemon treats', 'lemon-flavored candy'] ,  0.89636856
['conflict', 'drama'] ,  5.4892906e-05
['clothing worn on the feet', 'socks'] ,  0.0020660206
['throughout', 'always'] ,  3.7185084e-05
['a break', 'vacation'] ,  4.2760672e-05

['ice cream', 'fondue'] ,  3.9480252e-05
['warship', 'Statue of Liberty'] ,  1.4958433e-05
['birdfeed', 'the study of birds'] ,  0.00034959082
['staff made of magic', 'jazz-fusion'] ,  1.2440907e-05
```

## SentenceTransformer Model

In [83]:
ST_model = SentenceTransformer("all-MiniLM-L6-v2", device="mps" if torch.backends.mps.is_available() else "cpu")

loss = losses.CosineSimilarityLoss(ST_model)
print(ST_model)

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)


In [86]:
ST_model_save_path = "output/training_ST_" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# train the model
ST_model.fit(
    train_objectives=[(
        DataLoader(
            dataset=train_data,
            shuffle=True,
            batch_size=train_batch_size,
            pin_memory=True
        ),
        loss
    )],
    evaluator=evaluation.BinaryClassificationEvaluator.from_input_examples(cv_data),
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    output_path=ST_model_save_path
)

# write the config
with open(ST_model_save_path + "/config.txt", 'w') as f:
    f.write("train batch size: " + str(train_batch_size) + "\n")
    f.write("eval batch size: " + str(eval_batch_size) + "\n")
    f.write("num epochs: " + str(num_epochs) + "\n")
    f.write("warmup_steps: " + str(warmup_steps) + "\n")
    f.write("data_ratio: " + str(1 / data_ratio) + "\n")
    f.write("no dense layer")

In [99]:
positive = [
  ["place of origin", "birthplace"],
  ["alien", "foreigner"],
  ["warship", "naval war vessel"],
  ["lemon treats", "lemon-flavored candy"],
  ["conflict", "drama"],
  ["clothing worn on the feet","socks"],
  ["throughout","always"],
  ["a break", "vacation"]
]
negative = [
  ["ice cream", "fondue"],
  ["warship", "Statue of Liberty"],
  ["birdfeed", "the study of birds"],
  ["staff made of magic", "jazz-fusion"]
]

for [x, y] in positive:
  similarity = float(ST_model.similarity(ST_model.encode(x), ST_model.encode(y)))
  print([x, y], ", ", similarity)

for [x, y] in negative:
  similarity = float(ST_model.similarity(ST_model.encode(x), ST_model.encode(y)))
  print([x, y], ", ", similarity)


['place of origin', 'birthplace'] ,  0.8708105683326721
['alien', 'foreigner'] ,  0.9510957598686218
['warship', 'naval war vessel'] ,  0.7631083726882935
['lemon treats', 'lemon-flavored candy'] ,  0.8496531248092651
['conflict', 'drama'] ,  0.9787645936012268
['clothing worn on the feet', 'socks'] ,  0.851960301399231
['throughout', 'always'] ,  0.9862720966339111
['a break', 'vacation'] ,  0.9060254096984863
['ice cream', 'fondue'] ,  0.6539474725723267
['warship', 'Statue of Liberty'] ,  0.6794666051864624
['birdfeed', 'the study of birds'] ,  0.6723217368125916
['staff made of magic', 'jazz-fusion'] ,  0.3318568468093872


pre-training results:
```
['place of origin', 'birthplace'] ,  0.5736582279205322
['alien', 'foreigner'] ,  0.5416784286499023
['warship', 'naval war vessel'] ,  0.7769221663475037
['lemon treats', 'lemon-flavored candy'] ,  0.7464485764503479
['conflict', 'drama'] ,  0.47645804286003113
['clothing worn on the feet', 'socks'] ,  0.5724813938140869
['throughout', 'always'] ,  0.355945348739624
['a break', 'vacation'] ,  0.41463303565979004

['ice cream', 'fondue'] ,  0.4437253177165985
['warship', 'Statue of Liberty'] ,  0.3803083300590515
['birdfeed', 'the study of birds'] ,  0.5581598281860352
['staff made of magic', 'jazz-fusion'] ,  0.05649913102388382
```